## **Problem 2a**

Implement a suitable network architecture for classifying images of planes, ships, and trucks. Do **not** use pretrained models or library wrappers like `torchvision.models.vgg11`. We will build a **VGG-11** model using Table 1, Column A from Simonyan & Zisserman (2015), including **Batch Normalization** after each convolutional layer.

### Why VGG-11 with BatchNorm?

- **VGG-11** is a classic convolutional neural network architecture that uses relatively small (3×3) convolution filters, stacked in increasing depth.
- **Batch Normalization** (BN) helps stabilize and accelerate training by normalizing intermediate activations, reducing internal covariate shift, and allowing for higher learning rates.
- For a dataset with three classes (planes, ships, trucks) and images of size 96×96×3, VGG-11 is sufficiently expressive but not excessively large, making it a suitable baseline. BN generally yields better generalization and faster convergence.

Below, we provide the core PyTorch model definition for VGG-11 with BN.

In [ ]:
# %% [model.py]
# NOTE: This cell simulates the contents of a file named model.py
#       You can copy-paste it into an actual file if desired.

import torch
import torch.nn as nn
import torch.nn.functional as F

class VGG11BN(nn.Module):
    def __init__(self, num_classes=3, dropout=False):
        super(VGG11BN, self).__init__()
        self.num_classes = num_classes
        self.dropout = dropout

        # Convolutional part (features)
        # Configuration: 64 -> MP -> 128 -> MP -> 256 x2 -> MP -> 512 x2 -> MP -> 512 x2 -> MP
        # After each conv, we insert a batchnorm.
        # Pool kernel size is 2, stride 2.

        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Classifier part
        # Typically for VGG: 4096 -> 4096 -> num_classes
        # We'll apply dropout(0.5) in Problem 2c if self.dropout==True.

        self.classifier = nn.Sequential(
            nn.Linear(512 * 3 * 3, 4096),
            nn.ReLU(True),
            # Dropout to be optionally inserted:
            nn.Dropout(p=0.5) if self.dropout else nn.Identity(),

            nn.Linear(4096, 4096),
            nn.ReLU(True),
            # Dropout to be optionally inserted:
            nn.Dropout(p=0.5) if self.dropout else nn.Identity(),

            nn.Linear(4096, self.num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.classifier(x)
        return x


In [ ]:
# **Solution Code for Problem 2a**
# We instantiate our VGG-11 with BatchNorm (no dropout for now).

import torch

# Create a model instance for 3 classes (planes, ships, trucks) as required.
model_2a = VGG11BN(num_classes=3, dropout=False)
print(model_2a)

# This model is now defined and ready to be trained.
# In later problems, we will show how to train it and add Dropout.


## **Problem 2b**

Explain what **Dropout** is and why it is useful. Describe how Dropout works, providing a brief mathematical explanation if possible.

### Dropout Explanation

- **Definition**: Dropout randomly sets a fraction \(p\) of units (activations) to zero during training, with probability \(p\). This effectively thins the network, preventing co-adaptation of features.
- **Why it's useful**: Dropout acts as a regularizer and helps reduce overfitting, especially in fully connected layers that have many parameters.
- **How it works**:
  - During training, each neuron's output is kept with probability \(1 - p\), or set to zero with probability \(p\).
  - Mathematically, if \(h\) is a neuron's output, then \(\tilde{h} = r \cdot h\), where \(r\) is a random mask drawn from a Bernoulli distribution with parameter \(1 - p\).
  - At test time, no dropout is applied, but the layer outputs are scaled by the factor \((1 - p)\) to account for the average effect of dropout during training.


In [ ]:
# **Solution Code for Problem 2b**
# There's no direct computation to show here, but let's illustrate a simple Dropout example.

import torch.nn as nn
import torch

# Example: A small demonstration of dropout behavior on a random tensor
dropout_layer = nn.Dropout(p=0.5)
x = torch.ones(5)

print("Input:", x)
output_train = dropout_layer(x)
print("Output with Dropout (train mode):", output_train)

# Show that dropout is not applied in eval mode.
dropout_layer.eval()
output_eval = dropout_layer(x)
print("Output with Dropout (eval mode):", output_eval)


## **Problem 2c**

Modify the VGG-11 model to include **Dropout(p=0.5)** after each of the two fully connected layers with 4096 units. Provide the updated architecture in code and discuss its impact on training.

### Including Dropout

- We have already built an optional `dropout` argument into our `VGG11BN` class.
- By passing `dropout=True`, we place a `nn.Dropout(p=0.5)` after each 4096-unit linear layer.
- **Impact**: This helps reduce overfitting by randomly zeroing out neurons, making the network more robust but possibly slower to converge initially.


In [ ]:
# %% [utils.py]
# NOTE: This cell simulates the contents of a file named utils.py
#       It includes a custom Dataset class, a stratified split function, etc.

import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler
import matplotlib.pyplot as plt
from typing import Tuple, List

class ImageDataset(Dataset):
    def __init__(self, images: np.ndarray, labels: np.ndarray=None, transform=None):
        # images: (N, 96, 96, 3)
        # labels: (N, ) or None if unlabeled
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx].astype(np.float32)
        # Example: scale images to [0,1], convert to CHW
        img = img / 255.0
        img = np.transpose(img, (2, 0, 1))  # (3,96,96)

        if self.transform:
            # Apply any custom transforms (e.g. augmentations)
            img = self.transform(img)

        if self.labels is not None:
            label = self.labels[idx]
            return torch.tensor(img, dtype=torch.float), torch.tensor(label, dtype=torch.long)
        else:
            # For unlabeled data
            return torch.tensor(img, dtype=torch.float)

def split_dataset_stratified(images: np.ndarray, labels: np.ndarray,
                            val_fraction: float=0.2) -> Tuple[List[int], List[int]]:
    """
    Splits indices into train/val sets with class stratification.
    """
    # Basic example of stratified split:
    num_classes = len(np.unique(labels))
    train_indices = []
    val_indices = []

    for c in range(num_classes):
        c_indices = np.where(labels == c)[0]
        np.random.shuffle(c_indices)
        val_size = int(len(c_indices) * val_fraction)
        val_indices.extend(c_indices[:val_size])
        train_indices.extend(c_indices[val_size:])

    return train_indices, val_indices

def compute_entropy(probabilities: np.ndarray) -> float:
    """
    Computes the entropy H(y_hat) = - sum(p_i log(p_i)) for a 1D array of probabilities.
    """
    eps = 1e-9
    p = np.clip(probabilities, eps, 1.0)
    return -np.sum(p * np.log(p))

def plot_entropy_hist(entropies_known, entropies_unknown, bins=30):
    """
    Plots two histograms on the same figure.
    """
    plt.figure()
    plt.hist(entropies_known, bins=bins, alpha=0.5, label='Known Class Entropy')
    plt.hist(entropies_unknown, bins=bins, alpha=0.5, label='Unknown Class Entropy')
    plt.xlabel('Entropy')
    plt.ylabel('Frequency')
    plt.title('Entropy Histograms for Known vs. Unknown')
    plt.legend()
    plt.show()


In [ ]:
# %% [train.py]
# NOTE: This cell simulates the contents of a file named train.py
#       It contains a training loop, evaluation function, etc.

import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader

def evaluate_model(model, dataloader, criterion, device='cpu'):
    model.eval()
    running_loss = 0.0
    running_corrects = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            total_samples += inputs.size(0)

    epoch_loss = running_loss / total_samples
    epoch_acc = running_corrects.double() / total_samples
    return epoch_loss, epoch_acc.item()

def train_model(model, dataloaders, criterion, optimizer, scheduler=None, device='cpu', num_epochs=10):
    best_acc = 0.0
    best_model_wts = None

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 10)
        
        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0
            total_samples = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                total_samples += inputs.size(0)

            epoch_loss = running_loss / total_samples
            epoch_acc = running_corrects.double() / total_samples

            print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            # deep copy the model
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()

        if scheduler is not None:
            scheduler.step()
        print()

    print(f"Best val Acc: {best_acc:4f}")
    if best_model_wts is not None:
        model.load_state_dict(best_model_wts)
    return model


In [ ]:
# **Solution Code for Problem 2c**
# We'll instantiate the VGG-11 model with dropout and demonstrate how it might be trained.

import numpy as np
from torch.utils.data import DataLoader

# For demonstration, let's assume we have training_data.npz loaded:
# training_data = np.load('data/problem2/training_data.npz')
# training_images = training_data['a']  # shape (N, 96, 96, 3)
# training_labels = training_data['b']  # shape (N, )

# We'll create some dummy data for the sake of demonstration:
np.random.seed(42)
dummy_images = np.random.randint(0, 255, (100, 96, 96, 3), dtype=np.uint8)
dummy_labels = np.random.randint(0, 3, (100,), dtype=np.int64)

# Use our split_dataset_stratified:
from utils import split_dataset_stratified, ImageDataset

train_idx, val_idx = split_dataset_stratified(dummy_images, dummy_labels, val_fraction=0.2)

train_dataset = ImageDataset(dummy_images[train_idx], dummy_labels[train_idx])
val_dataset = ImageDataset(dummy_images[val_idx], dummy_labels[val_idx])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

dataloaders = {
    'train': train_loader,
    'val': val_loader
}

# Instantiate the model with dropout:
from model import VGG11BN
model_2c = VGG11BN(num_classes=3, dropout=True)

# Let's do a short training run to illustrate:
from train import train_model
import torch.nn as nn
import torch.optim as optim

device = 'cpu'  # or 'cuda' if available
model_2c = model_2c.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_2c.parameters(), lr=1e-4)

# We'll train for just 1 epoch in this demo
trained_model = train_model(
    model_2c,
    dataloaders,
    criterion,
    optimizer,
    scheduler=None,
    device=device,
    num_epochs=1
)

print("\nModel with Dropout training demo complete.")


## **Problem 2d**

Use the trained model (with Dropout enabled). For each image, perform 10 forward passes and compute the **softmax outputs**. Then calculate the **mean softmax** and the **entropy** of each sample:

$$\displaystyle H(\hat{y}) = - \sum_{i=1}^{N_c} \hat{y}_i \log(\hat{y}_i)$$

Plot a histogram of entropies for **known** vs. **unknown** classes. In markdown, explain why entropy reflects model uncertainty.

### Why Entropy Reflects Uncertainty
- The entropy of the predicted probability distribution measures how "spread out" the distribution is.
- A more peaked (confident) distribution results in lower entropy, while a more uniform (uncertain) distribution yields higher entropy.
- Hence, entropy is a natural measure of uncertainty in classification.

In [ ]:
# **Solution Code for Problem 2d**
# We'll simulate known vs unknown classes by using two small datasets.
# We'll do 10 forward passes for each sample with model_2c in train (dropout) mode.

import torch.nn.functional as F
from utils import compute_entropy, plot_entropy_hist

# Suppose we have some 'known' dataset (subset of the validation set) and some 'unknown' dataset.
known_data_loader = val_loader  # re-using from above for demonstration

# Let's create a small 'unknown' dataset by random images from a different distribution.
dummy_unknown_images = np.random.randint(0, 255, (20, 96, 96, 3), dtype=np.uint8)
unknown_dataset = ImageDataset(dummy_unknown_images, labels=None)
unknown_loader = DataLoader(unknown_dataset, batch_size=4, shuffle=False)

model_2c.train()  # Enable dropout

entropies_known = []
entropies_unknown = []

def mc_dropout_prediction(model, x, mc_passes=10):
    # Returns the mean softmax probability over mc_passes.
    # model is in train mode, so dropout is applied each time.
    probs_list = []
    for _ in range(mc_passes):
        with torch.no_grad():
            out = model(x)
            sm = F.softmax(out, dim=1)
            probs_list.append(sm.cpu().numpy())
    probs_mc = np.stack(probs_list, axis=0)  # shape: (mc_passes, batch_size, num_classes)
    mean_probs = np.mean(probs_mc, axis=0)  # shape: (batch_size, num_classes)
    return mean_probs

# Evaluate known data:
for inputs, labels in known_data_loader:
    inputs = inputs.to(device)
    mean_probs = mc_dropout_prediction(model_2c, inputs, mc_passes=10)
    # Compute entropy for each sample in the batch:
    for mp in mean_probs:
        ent = compute_entropy(mp)
        entropies_known.append(ent)

# Evaluate unknown data:
for inputs in unknown_loader:
    inputs = inputs.to(device)
    mean_probs = mc_dropout_prediction(model_2c, inputs, mc_passes=10)
    # Compute entropy for each sample in the batch:
    for mp in mean_probs:
        ent = compute_entropy(mp)
        entropies_unknown.append(ent)

# Now plot the histograms:
plot_entropy_hist(entropies_known, entropies_unknown)

print("Histogram plotted. Entropies computed for known vs. unknown.")


## **Problem 2e**

Use the entropies from 2d to determine a threshold. Report:
- How many **unwanted** class samples were correctly detected.
- How many **wanted** class samples were mistakenly removed.

### Discussion
- After computing the entropy for known and unknown sets, we can pick a threshold (e.g., a percentile of the known entropies).
- Samples with entropy above that threshold are flagged as "unknown," others as "known."

In [ ]:
# **Solution Code for Problem 2e**

import numpy as np

# Let entropies_known, entropies_unknown be from previous steps.
# We'll pick a threshold such that e.g., 95th percentile of known entropies.
threshold = np.percentile(entropies_known, 95)

print(f"Chosen entropy threshold: {threshold:.4f}")

# Classify known set:
known_predictions = [1 if e > threshold else 0 for e in entropies_known]
# 0 => predicted known, 1 => predicted unknown

# Classify unknown set:
unknown_predictions = [1 if e > threshold else 0 for e in entropies_unknown]

# How many unwanted class samples (truly unknown) were correctly detected?
correctly_detected_unknown = sum(unknown_predictions)  # 1 means flagged unknown

# How many wanted class samples (truly known) were mistakenly removed?
# i.e., how many known samples got predicted unknown?
mistakenly_removed_known = sum(known_predictions)

print(f"Unwanted class samples correctly detected: {correctly_detected_unknown}")
print(f"Wanted class samples mistakenly removed: {mistakenly_removed_known}")


## **Problem 2f (bonus)**

Apply your trained model to unlabeled images. Use the entropy threshold from 2e to classify images as known (0) or unknown (1). Store these binary predictions as a NumPy array and save using `.npz`. **Do not shuffle** the input data.


In [ ]:
# **Solution Code for Problem 2f**

# Suppose the unlabeled images have been loaded as:
# unlabeled_data = np.load('data/problem2/new_evaluation_data_without_labels.npz')
# unlabeled_images = unlabeled_data['a']

unlabeled_images = np.random.randint(0, 255, (10, 96, 96, 3), dtype=np.uint8)  # dummy data
unlabeled_dataset = ImageDataset(unlabeled_images, labels=None)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=2, shuffle=False)

model_2c.train()  # ensure dropout is enabled

binary_predictions = []

for inputs in unlabeled_loader:
    inputs = inputs.to(device)
    mean_probs = mc_dropout_prediction(model_2c, inputs, mc_passes=10)  # from above function
    # compute entropy
    for mp in mean_probs:
        ent = compute_entropy(mp)
        pred = 1 if ent > threshold else 0  # 1 => unknown, 0 => known
        binary_predictions.append(pred)

binary_predictions = np.array(binary_predictions)

# Save as .npz (not shuffling, preserving order)
np.savez('unlabeled_predictions.npz', predictions=binary_predictions)

print("Saved binary predictions to 'unlabeled_predictions.npz'.")
